In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.tuning import (
    load_training_data,
    split_data,
    tune_lightgbm,
    evaluate_model,
    save_model,
    save_best_parameters,
)

In [2]:
DATA_PATH = project_root / "data" / "preprocessed" / "train_selected.parquet"

print(DATA_PATH)

d:\Projects\credit-card-fraud-detection\data\preprocessed\train_selected.parquet


## Load Dataset

In [3]:
df = load_training_data(DATA_PATH)

df.head()


Loading training data...
            Shape : (590540, 79)


,TransactionAmt,ProductCD,card1,card4,card6,DeviceType,tx_hour,tx_day_of_week,tx_is_weekend,tx_is_night,...,M1,V221,V191,V171,V66,V7,id_35,id_29,id_20,isFraud
0,4.241327,4,13926,1,1,1,0,0,0,1,...,1.0,1.0,1.0,1.0,1.0,1.0,2,2,472.0,0
1,3.401197,4,2755,2,1,1,0,0,0,1,...,1.0,1.0,1.0,1.0,1.0,1.0,2,2,472.0,0
2,4.094345,4,4663,4,2,1,0,0,0,1,...,1.0,1.0,1.0,1.0,1.0,1.0,2,2,472.0,0
3,3.931826,4,18132,2,2,1,0,0,0,1,...,1.0,1.0,1.0,1.0,1.0,1.0,2,2,472.0,0
4,3.931826,1,4497,2,1,2,0,0,0,1,...,1.0,1.0,1.0,1.0,1.0,1.0,1,1,144.0,0


## Split Dataset

In [4]:
X_train, X_test, y_train, y_test = split_data(df)


Splitting dataset...
Train : (472432, 78)
Test  : (118108, 78)


## Hyperparameter Tuning

In [5]:
best_model, best_params = tune_lightgbm(X_train, y_train)


LIGHTGBM HYPERPARAMETER TUNING
Fitting 4 folds for each of 15 candidates, totalling 60 fits
[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.081718 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2395
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 77
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000

Best ROC-AUC :
0.8991586135495445

Best Parameters :
{'subsample': 0.7, 'num_leaves': 127, 'n_estimators': 700, 'min_child_samples': 20, 'max_depth': -1, 'learning_rate': 0.03, 'colsample_bytree': 0.7}


## Best Parameters

In [6]:
best_params

{'subsample': 0.7,
 'num_leaves': 127,
 'n_estimators': 700,
 'min_child_samples': 20,
 'max_depth': -1,
 'learning_rate': 0.03,
 'colsample_bytree': 0.7}

## Model Evaluation

In [7]:
results = evaluate_model(best_model, X_test, y_test)


MODEL EVALUATION
ROC-AUC   : 0.9069
Precision : 0.2417
Recall    : 0.7159
F1 Score  : 0.3614

Classification Report

              precision    recall  f1-score   support

           0       0.99      0.92      0.95    113975
           1       0.24      0.72      0.36      4133

    accuracy                           0.91    118108
   macro avg       0.62      0.82      0.66    118108
weighted avg       0.96      0.91      0.93    118108



## Save Model

In [8]:
save_model(best_model)


Model saved successfully!
Location : D:\Projects\credit-card-fraud-detection\models\lightgbm_tuned.pkl


## Save Best Parameters

In [9]:
save_best_parameters(best_params)


Best parameters saved!
D:\Projects\credit-card-fraud-detection\results\lightgbm_best_params.json


## Final Summary

In [10]:
print("="*60)
print("FINAL RESULTS")
print("="*60)

for key, value in results.items():
    if isinstance(value, float):
        print(f"{key:<15}: {value:.4f}")
    else:
        print(f"{key:<15}: {value}")

FINAL RESULTS
Model          : LightGBM (Tuned)
ROC-AUC        : 0.9069
Precision      : 0.2417
Recall         : 0.7159
F1 Score       : 0.3614
